# HouseholdBasketEnv — GRPO Training (Phases 6 + 7 + 8)

QLoRA + GRPO fine-tuning of **Qwen2.5-3B-Instruct** on `HouseholdBasketEnv`. This notebook runs:

- **MAIN** — full reward (default; plan §6)
- **Ablation A** — `enable_meal_type_coverage=False` (plan §7.A)
- **Ablation B** — train on Task 3 only, no curriculum mix (plan §7.B)

Switch experiments via the `RUN_NAME` cell at the top.

> **Hardware:** Local GPU or Colab T4/A100. T4 fits 3B in 4-bit + LoRA-r32. Expect ~6–8h for the main run at 800 steps.
>
> **Seeds:** drawn from `seeds/verified_task{2,3}.json` (training_seeds split). Held-out seeds are NEVER used during training.

## 1. Run config (edit me)

Choose ONE: `"main"`, `"ablation_a"`, or `"ablation_b"`. Outputs land in `runs/<RUN_NAME>/`.

In [21]:
# =============================================================================
# Pick exactly ONE run.
# =============================================================================
RUN_NAME = "ablation_b"            # "main" | "ablation_a" | "ablation_b"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Trainer hyperparameters (plan §6.4) ----------------------------------------
MAX_TRAIN_STEPS = 40
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
NUM_GENERATIONS = 8       # GRPO group size
GRPO_BETA = 0.04          # KL penalty
LEARNING_RATE = 5e-6
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 96
SAVE_EVERY = 200
SEED = 0

# LoRA / 4-bit ---------------------------------------------------------------
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LOAD_IN_4BIT = True
MAX_SEQ_LEN = 2048

# Curriculum mix (plan §6.2) -------------------------------------------------
TIER_WEIGHTS = {2: 0.7, 3: 0.3}   # default: main + ablation_a
if RUN_NAME == "ablation_b":
    TIER_WEIGHTS = {3: 1.0}

# Reward ablation (plan §7.A) ------------------------------------------------
ENABLE_MEAL_TYPE_COVERAGE = (RUN_NAME != "ablation_a")

print({
    "RUN_NAME": RUN_NAME,
    "TIER_WEIGHTS": TIER_WEIGHTS,
    "ENABLE_MEAL_TYPE_COVERAGE": ENABLE_MEAL_TYPE_COVERAGE,
    "MAX_TRAIN_STEPS": MAX_TRAIN_STEPS,
})

{'RUN_NAME': 'ablation_b', 'TIER_WEIGHTS': {3: 1.0}, 'ENABLE_MEAL_TYPE_COVERAGE': True, 'MAX_TRAIN_STEPS': 40}


## 2. Bootstrap (clone repo + install deps)

In [22]:
import os, sys, pathlib, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

CLONE_DIR = pathlib.Path("/app/hackathon/HouseholdBasketEnv")

REPO_ROOT = CLONE_DIR.resolve()
ENV_ROOT  = REPO_ROOT / "household_basket_env"
RESULTS_DIR = ENV_ROOT / "notebooks" / "results"
RUN_DIR = ENV_ROOT / "notebooks" / "runs" / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


os.environ["HOUSEHOLD_BASKET_PRODUCTS_PATH"] = str(ENV_ROOT / "data" / "products.json")
os.environ.setdefault("HF_HOME", str(REPO_ROOT / ".hf_cache"))
sys.path.insert(0, "/app/hackathon/HouseholdBasketEnv")

print("REPO_ROOT   =", REPO_ROOT)
print("ENV_ROOT    =", ENV_ROOT)
print("RESULTS_DIR =", RESULTS_DIR)

REPO_ROOT   = /app/hackathon/HouseholdBasketEnv
ENV_ROOT    = /app/hackathon/HouseholdBasketEnv/household_basket_env
RESULTS_DIR = /app/hackathon/HouseholdBasketEnv/household_basket_env/notebooks/results


In [23]:
os.environ["HTTP_PROXY"] = "http://azureproxy.jio.com:8080"
os.environ["HTTPS_PROXY"] = "http://azureproxy.jio.com:8080"

from household_basket_env.server.environment import HouseholdBasketEnvironment
from household_basket_env.models import BasketAction
from household_basket_env.server.curriculum import (
    TIER_CONFIGS,
    load_verified_seeds,
    TRAIN_TIER_WEIGHTS,
)
print("Env package imports OK.")

Env package imports OK.


## 3. Load Qwen2.5-3B-Instruct + LoRA

In [24]:
import os
import time
import threading
import torch


USE_UNSLOTH = True

# ── GPU Monitor ───────────────────────────────────────────────────────────────
stop_monitor = False
def monitor_gpu():
    prev = [0] * torch.cuda.device_count()
    while not stop_monitor:
        line = ""
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            used = (total - free) / 1e9
            delta = used - prev[i]
            arrow = f"▲+{delta:.2f}GB" if delta > 0.05 else ("▼" if delta < -0.05 else "─")
            pct = (used / total * 1e9) * 100
            line += f"  GPU{i}: {used:.2f}/{total/1e9:.1f}GB ({pct:.0f}%) {arrow} |"
            prev[i] = used
        print(f"  [VRAM] {line}")
        time.sleep(3)

monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()

# ── Pre-load snapshot ─────────────────────────────────────────────────────────
print("=" * 60)
print("🔍 PRE-LOAD STATE")
print("=" * 60)
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    used = (total - free) / 1e9
    print(f"  GPU {i}: {used:.2f} GB used / {total/1e9:.1f} GB total")
print("=" * 60)

t_total = time.time()

try:
    print("\n⏳ [Step 1/4] Importing Unsloth...")
    t = time.time()
    from unsloth import FastLanguageModel, PatchFastRL
    print(f"  ✅ Unsloth imported in {time.time()-t:.1f}s")

    print("\n⏳ [Step 2/4] Patching GRPO...")
    t = time.time()
    PatchFastRL("GRPO", FastLanguageModel)
    print(f"  ✅ GRPO patched in {time.time()-t:.1f}s")

    print("\n⏳ [Step 3/4] Loading base model weights (watch VRAM ▲)...")
    t = time.time()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=LOAD_IN_4BIT,
        fast_inference=False,
        max_lora_rank=LORA_R,
        gpu_memory_utilization=0.55,
    )
    elapsed = time.time() - t
    print(f"  ✅ Base model loaded in {elapsed:.1f}s")
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        used = (total - free) / 1e9
        print(f"  📊 GPU {i} after base load: {used:.2f} GB used / {total/1e9:.1f} GB")
    p = next(model.parameters())
    print(f"  📍 Model device : {p.device}")
    print(f"  📍 Model dtype  : {p.dtype}")

    print("\n⏳ [Step 4/4] Attaching LoRA adapters...")
    t = time.time()
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    print(f"  ✅ LoRA attached in {time.time()-t:.1f}s")

    # Trainable parameter count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  📊 Total params     : {total_params/1e6:.1f}M")
    print(f"  📊 Trainable params : {trainable_params/1e6:.1f}M ({100*trainable_params/total_params:.2f}%)")

    print(f"\n✅ Model + LoRA ready via Unsloth (total: {time.time()-t_total:.1f}s)")

except Exception as e:
    USE_UNSLOTH = False
    print(f"\n❌ Unsloth failed: {e}")
    print("   Falling back to Transformers + PEFT...\n")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model

    print("⏳ [Step 1/4] Loading tokenizer...")
    t = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    print(f"  ✅ Tokenizer loaded in {time.time()-t:.1f}s")

    print("\n⏳ [Step 2/4] Setting up BitsAndBytes config...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ) if LOAD_IN_4BIT else None
    print(f"  ✅ BnB config: {'4bit NF4' if bnb_config else 'disabled'}")

    print("\n⏳ [Step 3/4] Loading base model (watch VRAM ▲)...")
    t = time.time()
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"  ✅ Base model loaded in {time.time()-t:.1f}s")
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        used = (total - free) / 1e9
        print(f"  📊 GPU {i} after base load: {used:.2f} GB used / {total/1e9:.1f} GB")
    p = next(base.parameters())
    print(f"  📍 Model device : {p.device}")
    print(f"  📍 Model dtype  : {p.dtype}")

    print("\n⏳ [Step 4/4] Attaching LoRA adapters...")
    t = time.time()
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_config)
    model.gradient_checkpointing_enable()
    print(f"  ✅ LoRA attached in {time.time()-t:.1f}s")

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  📊 Total params     : {total_params/1e6:.1f}M")
    print(f"  📊 Trainable params : {trainable_params/1e6:.1f}M ({100*trainable_params/total_params:.2f}%)")

    print(f"\n✅ Model + LoRA ready via Transformers (total: {time.time()-t_total:.1f}s)")

finally:
    stop_monitor = True

print("\n" + "=" * 60)
print(f"🚀 Model ready! USE_UNSLOTH={USE_UNSLOTH}")
print("=" * 60)

🔍 PRE-LOAD STATE
  [VRAM]   GPU0: 24.06/85.1GB (28%) ▲+24.06GB |
  GPU 0: 24.06 GB used / 85.1 GB total

⏳ [Step 1/4] Importing Unsloth...
  ✅ Unsloth imported in 0.0s

⏳ [Step 2/4] Patching GRPO...
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
  ✅ GRPO patched in 0.0s

⏳ [Step 3/4] Loading base model weights (watch VRAM ▲).

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 880.77it/s]


  [VRAM]   GPU0: 25.95/85.1GB (30%) ▲+1.89GB |
  [VRAM]   GPU0: 25.95/85.1GB (30%) ─ |
  [VRAM]   GPU0: 25.95/85.1GB (30%) ─ |
  [VRAM]   GPU0: 25.95/85.1GB (30%) ─ |unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

  ✅ Base model loaded in 17.2s
  📊 GPU 0 after base load: 25.98 GB used / 85.1 GB
  📍 Model device : cuda:0
  📍 Model dtype  : torch.bfloat16

⏳ [Step 4/4] Attaching LoRA adapters...
  [VRAM]   GPU0: 25.98/85.1GB (31%) ─ |
  [VRAM]   GPU0: 25.98/85.1GB (31%) ─ |
  [VRAM]   GPU0: 26.02/85.1GB (31%) ─ |
  [VRAM]   GPU0: 26.07/85.1GB (31%) ─ |
  [VRAM]   GPU0: 26.21/85.1GB (31%) ▲+0.14GB |
  ✅ LoRA attached in 15.4s
  📊 Total params     : 1860.0M
  📊 Trainable params : 59.9M (3.22%)

✅ Model + LoRA ready via Unsloth (total: 32.6s)

🚀 Model ready! USE_UNSLOTH=True


## 4. Reward function

GRPO calls `reward_fn(prompts, completions)` once per group of `NUM_GENERATIONS`. For HouseholdBasketEnv we don't have a one-shot reward — we have an **episode** of up to `max_steps` actions. Strategy:

1. Each prompt encodes the **current observation only** for one specific (seed, step) tuple.
2. We use the model's completion as the action, step the env, and return the per-step reward.
3. GRPO group sampling explores diverse completions per state — exactly what we want for credit assignment in a sparse-terminal env.

This is a **per-step bandit** view of GRPO (matches Unsloth's GRPOTrainer + open-source recipes). For full episode-level credit, you can swap to TRL's PPOTrainer or a custom rollout loop; the per-step variant is sufficient for our reward shaping.

In [25]:
import json, re, random
from textwrap import dedent

SYSTEM_PROMPT = """You are a careful nutrition-aware shopping assistant for an Indian household. \
At each step you pick exactly ONE product from the catalog and assign it to ONE household member. \
You MUST output ONLY a JSON object with three keys: \"product_id\" (string), \"member_id\" (string), \"rationale\" (1 short sentence). \
Do NOT add any prose, code fences, or extra keys. Respect each member's caps and allergies; spread meal types across breakfast/lunch/dinner/snack/beverage; stay under the budget."""

JSON_RE = re.compile(r"\{[^{}]*\}", re.S)

def render_observation(obs) -> str:
    members = []
    for m in obs.household:
        caps = ", ".join(f"{k}={v:.0f}" for k, v in m.thresholds_cap.items())
        members.append(f"- {m.member_id} | conds={m.conditions} | caps: {caps}")
    cands = []
    for c in obs.candidates[:50]:
        nuts = c.nutrition_per_100g
        cands.append(
            f"- {c.product_id} | {c.category} | meal={c.meal_type} | INR {c.price_inr:.0f} | "
            f"sugars_g={nuts.get('sugars_g',0):.1f} sodium_mg={nuts.get('sodium_mg',0):.0f} fat_g={nuts.get('fat_g',0):.1f}"
        )
    basket = [f"- {t.product_id} -> {t.member_id}" for t in obs.basket_so_far]
    return dedent(f"""
    Household:
    {chr(10).join(members)}

    Catalog (top 50):
    {chr(10).join(cands)}

    Basket so far ({len(basket)}):
    {chr(10).join(basket) if basket else "(empty)"}

    Budget remaining: INR {obs.budget_remaining:.0f}
    Steps used: {obs.step_index} / {obs.max_steps}
    Attempts used: {obs.attempt_index} / {obs.max_steps * 4}

    Now pick ONE product and assign it to ONE member. Output JSON only.
    """).strip()

def extract_json(text: str):
    m = JSON_RE.search(text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

print("Prompt utilities ready.")

Prompt utilities ready.


In [26]:
# Build per-step training prompts from the verified seed pool ----------------
import itertools

train_pool = []  # list of (seed, task_id)
for tier_id, w in TIER_WEIGHTS.items():
    payload = load_verified_seeds(tier_id)
    for s in payload["training_seeds"]:
        train_pool.append((s, tier_id))

if not train_pool:
    raise RuntimeError("Empty training pool. Re-run seed_verifier.")

print(f"Training pool: {len(train_pool)} (seed, tier) pairs across tiers {sorted(TIER_WEIGHTS.keys())}.")

# A persistent env that we reset per-step ------------------------------------
TRAIN_ENV = HouseholdBasketEnvironment()

def sample_starting_observation(rng: random.Random):
    """Pick a (seed, tier) by tier weight, reset env, return rendered prompt + obs ref."""
    tiers = list(TIER_WEIGHTS.keys())
    weights = [TIER_WEIGHTS[t] for t in tiers]
    tier_id = rng.choices(tiers, weights=weights, k=1)[0]
    payload = load_verified_seeds(tier_id)
    seed = rng.choice(payload["training_seeds"])
    obs = TRAIN_ENV.reset(seed=seed, task_id=tier_id)
    return seed, tier_id, obs

# Build a prompt set for the trainer. GRPOTrainer expects a HF Dataset of prompts.
import datasets

def make_prompt_record(rng: random.Random):
    seed, tier_id, obs = sample_starting_observation(rng)
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": render_observation(obs)},
        ],
        "seed": seed,
        "task_id": tier_id,
    }

# Generate 4096 prompt seeds; the trainer iterates with replacement.
PROMPT_RNG = random.Random(SEED)
prompt_rows = [make_prompt_record(PROMPT_RNG) for _ in range(4096)]
prompt_ds = datasets.Dataset.from_list(prompt_rows)
print("Prompt dataset:", prompt_ds)

Training pool: 70 (seed, tier) pairs across tiers [3].
Prompt dataset: Dataset({
    features: ['prompt', 'seed', 'task_id'],
    num_rows: 4096
})


In [27]:
# --- Reward function (per-step bandit view of GRPO) -------------------------
# We replay the env to the prompt's starting state, then score the completion.

from household_basket_env.server.environment import HouseholdBasketEnvironment

# Use a thread-local env so concurrent reward evaluation doesn't cross-talk.
def reward_basket_step(prompts, completions, **kwargs):
    """Returns one float per (prompt, completion). Plan §4 dense rewards.

    The trainer passes `seed` and `task_id` columns through `**kwargs` thanks to
    GRPOTrainer's `remove_unused_columns=False` flag below.
    """
    seeds = kwargs["seed"]
    task_ids = kwargs["task_id"]
    rewards = []
    env = HouseholdBasketEnvironment(enable_meal_type_coverage=ENABLE_MEAL_TYPE_COVERAGE)
    for completion, seed, task_id in zip(completions, seeds, task_ids):
        env.reset(seed=int(seed), task_id=int(task_id))
        text = completion[0]["content"] if isinstance(completion, list) else completion
        payload = extract_json(text)
        if payload is None:
            payload = text  # P_parse path
        out = env.apply_raw_action(payload)
        rewards.append(float(out.reward or 0.0))
    return rewards

# Fall back: the env may not yet support the kwarg. If so, monkey-patch.
import inspect
sig = inspect.signature(HouseholdBasketEnvironment.__init__)
if "enable_meal_type_coverage" not in sig.parameters:
    print("[warn] enable_meal_type_coverage kwarg not in env; ablation_a will silently no-op.")
print("Reward fn ready.")

Reward fn ready.


## 5. GRPOTrainer

In [28]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(RUN_DIR),
    learning_rate=LEARNING_RATE,
    beta=GRPO_BETA,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_prompt_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
    max_completion_length=MAX_NEW_TOKENS,
    max_steps=MAX_TRAIN_STEPS,
    save_steps=SAVE_EVERY,
    logging_steps=10,
    seed=SEED,
    bf16=True,
    fp16=False, 
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    temperature=TEMPERATURE,
    remove_unused_columns=False,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_basket_step],
    args=config,
    train_dataset=prompt_ds,
)
print("Trainer constructed.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer constructed.


In [29]:
history = trainer.train()
print("Training finished.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,096 | Num Epochs = 1 | Total steps = 40
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/app/hackathon/HouseholdBasketEnv/.venv/lib64/python3.12/site-packages/transformers/modeli

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_basket_step / mean,rewards / reward_basket_step / std
10,0.000263,-0.239683,0.029180,82.575000,38.000000,96.000000,0.750000,49.550000,38.000000,62.500000,0.006583,-0.239683,0.029180
20,0.000699,-0.239955,0.028412,81.912500,32.200000,96.000000,0.787500,22.600000,13.000000,34.500000,0.017487,-0.239955,0.028412
30,0.000689,-0.239914,0.028527,78.225000,22.300000,96.000000,0.675000,39.841667,22.300000,59.200000,0.017232,-0.239914,0.028527
40,0.000555,-0.250000,0.000000,85.337500,32.800000,96.000000,0.787500,45.283334,32.800000,57.200000,0.013874,-0.250000,0.000000


[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Training finished.


## 6. Save checkpoints + training log

In [30]:
import json

ADAPTER_DIR = RUN_DIR / "adapter_final"
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved final adapters ->", ADAPTER_DIR)

log_path = RUN_DIR / "training_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2)
print("Saved log ->", log_path)

config_path = RUN_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "RUN_NAME": RUN_NAME,
        "MODEL_NAME": MODEL_NAME,
        "TIER_WEIGHTS": TIER_WEIGHTS,
        "ENABLE_MEAL_TYPE_COVERAGE": ENABLE_MEAL_TYPE_COVERAGE,
        "MAX_TRAIN_STEPS": MAX_TRAIN_STEPS,
        "PER_DEVICE_BATCH": PER_DEVICE_BATCH,
        "GRAD_ACCUM": GRAD_ACCUM,
        "NUM_GENERATIONS": NUM_GENERATIONS,
        "GRPO_BETA": GRPO_BETA,
        "LEARNING_RATE": LEARNING_RATE,
        "TEMPERATURE": TEMPERATURE,
        "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        "LORA_R": LORA_R,
        "LORA_ALPHA": LORA_ALPHA,
        "LORA_DROPOUT": LORA_DROPOUT,
        "LOAD_IN_4BIT": LOAD_IN_4BIT,
    }, f, indent=2)
print("Saved config ->", config_path)

[transformers] Unsloth: Restored added_tokens_decoder metadata in /app/hackathon/HouseholdBasketEnv/household_basket_env/notebooks/runs/ablation_b/adapter_final/tokenizer_config.json.


Saved final adapters -> /app/hackathon/HouseholdBasketEnv/household_basket_env/notebooks/runs/ablation_b/adapter_final
Saved log -> /app/hackathon/HouseholdBasketEnv/household_basket_env/notebooks/runs/ablation_b/training_log.json
Saved config -> /app/hackathon/HouseholdBasketEnv/household_basket_env/notebooks/runs/ablation_b/run_config.json


### Next steps

- For each ablation, restart the notebook with a different `RUN_NAME` cell value.
- Run `eval_and_plots.ipynb` to compare against `docs/results/baseline.json`.
- Optional: push `adapter_final/` to HuggingFace via `scripts/push_to_hf.sh`.